# MIA5100 — Assignment 1
**Dataset:** MedicalCentre.csv (Parts 1–3) · housing.csv (Part 4)

| Part | Topic | Marks |
|---|---|---|
| 1 | Data Wrangling & Feature Engineering | 30 |
| 2 | Model Development & Evaluation (Naïve Bayes) | 20 |
| 3 | Model Comparison (NB vs Ensemble) | 30 |
| 4a | Unsupervised Learning — K-Means Clustering | 20 |
| 4b | Unsupervised Learning — DBSCAN Analysis | 20 |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')

---
# Part 1 — Data Wrangling & Feature Engineering  (30 marks)
**Dataset:** `MedicalCentre.csv`

---
## 1. Load and Explore the Dataset

In [ ]:
df = pd.read_csv('MedicalCentre.csv')
print(f"Dataset shape: {df.shape}")
df.head(10)

In [ ]:
print("Column data types:")
print(df.dtypes)
print("\nBasic statistics:")
df.describe(include='all')

---
## 2. Data Preparation (Missing Values, Duplicates, Irrelevant Attributes)

In [ ]:
# --- Missing values ---
missing_counts = df.isnull().sum()
missing_pct    = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_counts,
                            'Missing %': missing_pct.round(2)})
print("Missing values per column:")
print(missing_df)
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# --- Duplicates ---
print(f"Duplicate rows            : {df.duplicated().sum()}")
print(f"Duplicate AppointmentIDs  : {df['AppointmentID'].duplicated().sum()}")
print(f"Duplicate PatientIDs      : {df['PatientID'].duplicated().sum()}")

In [ ]:
# Drop duplicate rows
df = df.drop_duplicates()
print(f"Shape after dropping duplicates: {df.shape}")

# Drop irrelevant attributes (PatientID and AppointmentID are identifiers, not predictive features)
df = df.drop(columns=['PatientID', 'AppointmentID'])
print(f"Shape after dropping ID columns : {df.shape}")

# Drop rows with missing Age
print(f"\nRows with missing Age: {df['Age'].isnull().sum()}")
df = df.dropna(subset=['Age'])
print(f"Shape after dropping missing Age rows: {df.shape}")

---
## 3. Outlier Detection and Visualization

In [ ]:
# Boxplots for all numeric columns to visualize outliers
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns: {numeric_cols}")

n_cols   = 3
n_rows   = (len(numeric_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[i].set_title(col, fontsize=12)
    axes[i].set_ylabel('Value')

for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Boxplots for Outlier Detection', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

---
## 4. Negative Age Observations

In [ ]:
# Count frequency of negative Age observations
negative_ages = df[df['Age'] < 0]['Age']
print(f"Number of negative Age observations: {len(negative_ages)}")
print("\nFrequency of each negative Age value:")
print(negative_ages.value_counts().sort_index())

In [ ]:
# Remove negative Age values
df = df[df['Age'] >= 0].copy()
print(f"Shape after removing negative Age values: {df.shape}")
print(f"Age range after cleaning: {df['Age'].min()} – {df['Age'].max()}")

---
## 5. Create AgeGroup Feature

In [ ]:
# Bin mapping: {'0-1':1, '2-5':2, '6-10':3, '11-15':4, '16-25':5,
#               '26-30':6, '31-35':7, '36-40':8, '41-50':9, '51-115':10}
age_bins   = [0, 1, 5, 10, 15, 25, 30, 35, 40, 50, 115]
age_labels = [1, 2, 3,  4,  5,  6,  7,  8,  9,  10]

df['AgeGroup'] = pd.cut(df['Age'], bins=age_bins,
                        labels=age_labels, include_lowest=True)
df['AgeGroup'] = df['AgeGroup'].astype(int)

print("AgeGroup value distribution:")
print(df['AgeGroup'].value_counts().sort_index())

---
## 6. Rescale Age Using Min-Max Normalization

In [ ]:
scaler = MinMaxScaler()
df['Age_normalized'] = scaler.fit_transform(df[['Age']])

print("Original Age statistics:")
print(df['Age'].describe())
print("\nNormalized Age statistics:")
print(df['Age_normalized'].describe())

# Drop original Age column
df = df.drop(columns=['Age'])
print(f"\nShape after dropping original Age: {df.shape}")

---
## 7. Derive and Clean the AwaitingTime Feature

In [ ]:
# Parse dates
df['ScheduledDay']   = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])

# AwaitingTime = AppointmentDay − ScheduledDay (in whole days)
df['AwaitingTime'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days

print("AwaitingTime statistics BEFORE handling negatives:")
print(df['AwaitingTime'].describe())
print(f"\nNegative AwaitingTime values: {(df['AwaitingTime'] < 0).sum()}")

# Transform negative values to positive using absolute value
df['AwaitingTime'] = df['AwaitingTime'].abs()

print("\nAwaitingTime statistics AFTER handling negatives:")
print(df['AwaitingTime'].describe())

---
## 8. Create AwaitingTimeGroup Feature

In [ ]:
# Check unique values
print("Unique AwaitingTime values (sorted):")
unique_wait = sorted(df['AwaitingTime'].unique())
print(unique_wait)
print(f"\nMin: {df['AwaitingTime'].min()}, Max: {df['AwaitingTime'].max()}")

In [ ]:
# Bin mapping: {'0-1 days':1, '1-7 days':2, '8-30 days':3, '31-90 days':4, '91-180 days':5}
# Use max+1 as upper bound to ensure all values are captured
max_wait = int(df['AwaitingTime'].max())
wait_bins   = [0, 1, 7, 30, 90, max(180, max_wait)]
wait_labels = [1, 2, 3,  4,  5]

df['AwaitingTimeGroup'] = pd.cut(df['AwaitingTime'], bins=wait_bins,
                                  labels=wait_labels, include_lowest=True)
df['AwaitingTimeGroup'] = df['AwaitingTimeGroup'].astype(int)

print("AwaitingTimeGroup distribution:")
print(df['AwaitingTimeGroup'].value_counts().sort_index())

# Drop original AwaitingTime column
df = df.drop(columns=['AwaitingTime'])
print(f"\nShape after dropping AwaitingTime: {df.shape}")

---
## 9. Separate Date Features into Components

In [ ]:
# ScheduledDay components
df['ScheduledDay_year']    = df['ScheduledDay'].dt.year
df['ScheduledDay_month']   = df['ScheduledDay'].dt.month
df['ScheduledDay_day']     = df['ScheduledDay'].dt.day
df['ScheduledDay_hour']    = df['ScheduledDay'].dt.hour
df['ScheduledDay_weekday'] = df['ScheduledDay'].dt.weekday   # 0=Monday … 6=Sunday

# AppointmentDay components
df['AppointmentDay_year']    = df['AppointmentDay'].dt.year
df['AppointmentDay_month']   = df['AppointmentDay'].dt.month
df['AppointmentDay_day']     = df['AppointmentDay'].dt.day
df['AppointmentDay_weekday'] = df['AppointmentDay'].dt.weekday

# Drop original date columns
df = df.drop(columns=['ScheduledDay', 'AppointmentDay'])

print(f"Shape after date decomposition: {df.shape}")
df.head()

---
## 10. Encode Categorical Features

In [ ]:
# Identify remaining string/object categorical columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical columns to encode: {cat_cols}")

for col in cat_cols:
    cat = pd.Categorical(df[col])
    print(f"  {col}: {dict(enumerate(cat.categories))}")
    df[col] = cat.codes

print(f"\nShape after encoding: {df.shape}")
print("\nData types after encoding:")
print(df.dtypes)

---
## 11. Variability Comparison

In [ ]:
# Compute variance, standard deviation, and coefficient of variation (CV)
variability = pd.DataFrame({
    'Variance': df.var(),
    'Std Dev' : df.std(),
    'CV (%)'  : (df.std() / df.mean().abs() * 100).round(2)
}).sort_values('Variance', ascending=False)

print("Variability comparison across all features:")
print(variability.to_string())

In [ ]:
# Bar chart of variance per feature
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

variability['Variance'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Feature Variance', fontsize=14)
axes[0].set_xlabel('Feature')
axes[0].set_ylabel('Variance')
axes[0].tick_params(axis='x', rotation=45)

variability['Std Dev'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Feature Standard Deviation', fontsize=14)
axes[1].set_xlabel('Feature')
axes[1].set_ylabel('Standard Deviation')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Variability Comparison Across Features', fontsize=16)
plt.tight_layout()
plt.show()

---
## 12. Final Dataset Overview

In [ ]:
print(f"Final dataset shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())
print("\nFinal data types:")
print(df.dtypes)
print("\nFinal statistics:")
df.describe()

In [ ]:
df.head(10)

---
# Part 2 — Model Development & Evaluation  (20 marks)
**Classifier:** Gaussian Naïve Bayes · 70/30 stratified split

---
## 13. Model Development — Naive Bayes Classifier

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

# Separate features and target
X = df.drop(columns=['No-show'])
y = df['No-show']

print(f"Feature matrix : {X.shape}")
print(f"Target column  : {y.name}")
print(f"\nClass distribution:")
print(y.value_counts())
print(f"\nClass balance (%):")
print((y.value_counts(normalize=True) * 100).round(2))


### 14.1 Train / Test Split (70 % / 30 %)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"\nTrain class balance:\n{y_train.value_counts(normalize=True).round(4)}")
print(f"\nTest class balance :\n{y_test.value_counts(normalize=True).round(4)}")


### 14.2 Train the Gaussian Naïve Bayes Model

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)

y_pred_test  = gnb.predict(X_test)
y_pred_train = gnb.predict(X_train)

print("GaussianNB model trained successfully.")
print(f"var_smoothing (default): {gnb.var_smoothing}")


---
## 15. Performance Evaluation on the Test Set

In [ ]:
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)

y_prob_test = gnb.predict_proba(X_test)[:, 1]

print("=" * 55)
print("NAIVE BAYES — TEST SET PERFORMANCE")
print("=" * 55)
print(f"\nAccuracy : {accuracy_score(y_test, y_pred_test):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_test):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_test,
                             target_names=['Show (0)', 'No-show (1)']))


In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred_test)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Show', 'No-show'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix — Test Set', fontsize=13)

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
auc_score   = roc_auc_score(y_test, y_prob_test)
axes[1].plot(fpr, tpr, color='steelblue', lw=2,
             label=f'GaussianNB (AUC = {auc_score:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Test Set', fontsize=13)
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


---
## 16. Overfitting Check — Train vs Test Performance

In [ ]:
train_acc = accuracy_score(y_train, y_pred_train)
test_acc  = accuracy_score(y_test,  y_pred_test)

print("=" * 55)
print("OVERFITTING CHECK — Train vs Test")
print("=" * 55)
print(f"\nTraining Accuracy : {train_acc:.4f}")
print(f"Test Accuracy     : {test_acc:.4f}")
print(f"Accuracy Gap      : {abs(train_acc - test_acc):.4f}")
print("\n(A small gap indicates the model generalises well; a large gap")
print(" indicates overfitting.)")


In [ ]:
print("--- Training Set Classification Report ---")
print(classification_report(y_train, y_pred_train,
                             target_names=['Show (0)', 'No-show (1)']))
print("--- Test Set Classification Report ---")
print(classification_report(y_test, y_pred_test,
                             target_names=['Show (0)', 'No-show (1)']))


In [ ]:
# Side-by-side per-class metric bar charts
import pandas as pd

rep_train = pd.DataFrame(
    classification_report(y_train, y_pred_train, output_dict=True)
).T.iloc[:2]
rep_test  = pd.DataFrame(
    classification_report(y_test, y_pred_test, output_dict=True)
).T.iloc[:2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (title, report) in zip(axes, [('Train Set', rep_train),
                                        ('Test Set',  rep_test)]):
    report[['precision', 'recall', 'f1-score']].plot(
        kind='bar', ax=ax, edgecolor='black')
    ax.set_title(f'{title} — Per-Class Metrics', fontsize=13)
    ax.set_xticklabels(['Show (0)', 'No-show (1)'], rotation=0)
    ax.set_ylim(0, 1.1)
    ax.legend(loc='lower right')

plt.suptitle('Train vs Test Performance Comparison', fontsize=15)
plt.tight_layout()
plt.show()


---
## 17. Hyperparameter Tuning with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# GaussianNB's only tunable parameter is var_smoothing
param_grid = {'var_smoothing': np.logspace(-11, 0, 13)}
print("Search grid:")
print({k: [f'{v:.1e}' for v in vals] for k, vals in param_grid.items()})

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator  = GaussianNB(),
    param_grid = param_grid,
    cv         = cv_strategy,
    scoring    = 'roc_auc',
    n_jobs     = -1,
    verbose    = 1
)
grid_search.fit(X_train, y_train)

print(f"\nBest var_smoothing : {grid_search.best_params_['var_smoothing']:.2e}")
print(f"Best CV ROC-AUC    : {grid_search.best_score_:.4f}")


In [ ]:
# Evaluate the tuned model on the test set
best_gnb = grid_search.best_estimator_
y_pred_tuned = best_gnb.predict(X_test)
y_prob_tuned = best_gnb.predict_proba(X_test)[:, 1]

print("=" * 55)
print("TUNED MODEL — TEST SET PERFORMANCE")
print("=" * 55)
print(f"\nAccuracy : {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_tuned):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned,
                             target_names=['Show (0)', 'No-show (1)']))


In [ ]:
# Plot CV scores across the var_smoothing grid
cv_results = grid_search.cv_results_
mean_scores = cv_results['mean_test_score']
std_scores  = cv_results['std_test_score']
params      = param_grid['var_smoothing']

plt.figure(figsize=(10, 5))
plt.semilogx(params, mean_scores, marker='o', color='steelblue', lw=2,
             label='Mean CV ROC-AUC')
plt.fill_between(params, mean_scores - std_scores, mean_scores + std_scores,
                 alpha=0.2, color='steelblue', label='±1 Std Dev')
plt.axvline(grid_search.best_params_['var_smoothing'], color='red',
            linestyle='--',
            label=f"Best: {grid_search.best_params_['var_smoothing']:.2e}")
plt.xlabel('var_smoothing (log scale)')
plt.ylabel('Mean CV ROC-AUC')
plt.title('GridSearchCV — GaussianNB var_smoothing Tuning', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()


---
## 18. Model Summary

In [ ]:
summary = pd.DataFrame({
    'Model'   : ['GaussianNB (default)', 'GaussianNB (tuned)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_test),
        accuracy_score(y_test, y_pred_tuned)
    ],
    'ROC-AUC' : [
        roc_auc_score(y_test, y_prob_test),
        roc_auc_score(y_test, y_prob_tuned)
    ],
    'var_smoothing': [gnb.var_smoothing,
                      grid_search.best_params_['var_smoothing']]
})
print(summary.to_string(index=False))


---
# Part 3 — Model Comparison  (30 marks)
**Ensemble models:** Random Forest · XGBoost · compared against Naïve Bayes on same split

---
## 19. Ensemble Model Training

Using the **same** 70/30 train-test split from Section 14.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ── Random Forest ──────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf  = rf.predict(X_test)
y_prob_rf  = rf.predict_proba(X_test)[:, 1]
print("Random Forest trained.")
print(f"  n_estimators : {rf.n_estimators}")
print(f"  max_features : {rf.max_features}")

# ── XGBoost ────────────────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators  = 100,
    max_depth     = 6,
    learning_rate = 0.1,
    random_state  = 42,
    n_jobs        = -1,
    eval_metric   = 'logloss',
    verbosity     = 0
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]
print("\nXGBoost trained.")
print(f"  n_estimators  : {xgb.n_estimators}")
print(f"  max_depth     : {xgb.max_depth}")
print(f"  learning_rate : {xgb.learning_rate}")


In [ ]:
# Individual performance reports for each ensemble model
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

for name, y_pred, y_prob in [
    ('Random Forest', y_pred_rf,  y_prob_rf),
    ('XGBoost',       y_pred_xgb, y_prob_xgb),
]:
    print(f"{'=' * 55}")
    print(f"{name.upper()} — TEST SET PERFORMANCE")
    print(f"{'=' * 55}")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")
    print(classification_report(y_test, y_pred,
                                 target_names=['Show (0)', 'No-show (1)']))

# Confusion matrices side-by-side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, y_pred) in zip(axes, [('Random Forest', y_pred_rf),
                                       ('XGBoost',       y_pred_xgb)]):
    cm   = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Show', 'No-show'])
    disp.plot(ax=ax, cmap='Oranges', colorbar=False)
    ax.set_title(f'Confusion Matrix — {name}', fontsize=13)
plt.tight_layout()
plt.show()


---
## 20. Model Comparison — Accuracy, Sensitivity, Specificity & F1

In [ ]:
from sklearn.metrics import f1_score

def compute_metrics(model_name, y_true, y_pred, y_prob):
    cm           = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()
    return {
        'Model'      : model_name,
        'Accuracy'   : accuracy_score(y_true, y_pred),
        'Sensitivity': TP / (TP + FN),          # True Positive Rate
        'Specificity': TN / (TN + FP),          # True Negative Rate
        'F1 Score'   : f1_score(y_true, y_pred, pos_label=1),
        'ROC-AUC'    : roc_auc_score(y_true, y_prob),
    }

all_metrics = [
    compute_metrics('GaussianNB (default)', y_test, y_pred_test,  y_prob_test),
    compute_metrics('GaussianNB (tuned)',   y_test, y_pred_tuned, y_prob_tuned),
    compute_metrics('Random Forest',        y_test, y_pred_rf,    y_prob_rf),
    compute_metrics('XGBoost',              y_test, y_pred_xgb,   y_prob_xgb),
]

comparison_df = (pd.DataFrame(all_metrics)
                   .set_index('Model')
                   .round(4))

print("=" * 75)
print("MODEL COMPARISON TABLE")
print("=" * 75)
print(comparison_df.to_string())


In [ ]:
# Best and worst per metric
metrics_list = ['Accuracy', 'Sensitivity', 'Specificity', 'F1 Score', 'ROC-AUC']
print("\n" + "=" * 75)
print("BEST & WORST MODEL PER CRITERION")
print("=" * 75)
print(f"{'Metric':<15}  {'Best Model':<26}  {'Score':>6}    {'Worst Model':<26}  {'Score':>6}")
print("-" * 75)
for m in metrics_list:
    best  = comparison_df[m].idxmax()
    worst = comparison_df[m].idxmin()
    print(f"{m:<15}  {best:<26}  {comparison_df.loc[best,  m]:>6.4f}    "
          f"{worst:<26}  {comparison_df.loc[worst, m]:>6.4f}")


In [ ]:
# Grouped bar chart — all metrics across all models
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: Accuracy, Sensitivity, Specificity, F1
comparison_df[['Accuracy','Sensitivity','Specificity','F1 Score']].plot(
    kind='bar', ax=axes[0], edgecolor='black', width=0.7)
axes[0].set_title('Accuracy / Sensitivity / Specificity / F1 Score', fontsize=13)
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.15)
axes[0].set_xticklabels(comparison_df.index, rotation=15, ha='right')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].axhline(0.5, color='grey', linestyle='--', lw=0.8)

# Right: ROC-AUC only (zoomed for clarity)
comparison_df[['ROC-AUC']].plot(
    kind='bar', ax=axes[1], color='mediumpurple', edgecolor='black', width=0.4)
axes[1].set_title('ROC-AUC Comparison', fontsize=13)
axes[1].set_ylabel('AUC Score')
axes[1].set_ylim(0.60, 0.80)
axes[1].set_xticklabels(comparison_df.index, rotation=15, ha='right')
axes[1].legend(loc='upper right')
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.002,
                 f'{bar.get_height():.4f}',
                 ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Performance Comparison', fontsize=15)
plt.tight_layout()
plt.show()


---
## 21. ROC Analysis — All Models on a Single Graph

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

model_roc_data = [
    ('GaussianNB (default)', y_prob_test,  'steelblue',    '-'),
    ('GaussianNB (tuned)',   y_prob_tuned, 'cornflowerblue','--'),
    ('Random Forest',        y_prob_rf,    'darkorange',    '-'),
    ('XGBoost',              y_prob_xgb,   'green',         '-'),
]

for name, prob, color, ls in model_roc_data:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc         = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, color=color, linestyle=ls, lw=2.2,
            label=f'{name}  (AUC = {auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier (AUC = 0.5000)')
ax.fill_between([0,1], [0,1], alpha=0.05, color='grey')

ax.set_xlabel('False Positive Rate  (1 − Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate  (Sensitivity)',      fontsize=12)
ax.set_title('ROC Curves — Naïve Bayes vs Ensemble Models', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.show()


In [ ]:
# AUC summary table + interpretation
auc_summary = comparison_df[['ROC-AUC']].copy()
auc_summary['Discrimination'] = auc_summary['ROC-AUC'].apply(
    lambda v: 'Excellent (>0.9)' if v > 0.9
    else 'Good (0.8–0.9)' if v > 0.8
    else 'Fair (0.7–0.8)' if v > 0.7
    else 'Poor (0.6–0.7)' if v > 0.6
    else 'Fail (≤0.5)')

print("=" * 55)
print("AUC DISCRIMINATION ANALYSIS")
print("=" * 55)
print(auc_summary.to_string())
print()
best_auc  = auc_summary['ROC-AUC'].idxmax()
worst_auc = auc_summary['ROC-AUC'].idxmin()
print(f"Best discrimination  : {best_auc} (AUC = {auc_summary.loc[best_auc,  'ROC-AUC']:.4f})")
print(f"Worst discrimination : {worst_auc} (AUC = {auc_summary.loc[worst_auc, 'ROC-AUC']:.4f})")


In [ ]:
# Zoomed ROC — upper-left corner (high-sensitivity region)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (xlim, ylim, title) in zip(axes, [
    ((0, 1),    (0, 1.02), 'Full ROC Curves'),
    ((0, 0.5),  (0.5, 1),  'Zoomed — High-Sensitivity Region'),
]):
    for name, prob, color, ls in model_roc_data:
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc         = roc_auc_score(y_test, prob)
        ax.plot(fpr, tpr, color=color, linestyle=ls, lw=2,
                label=f'{name}  ({auc:.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(title, fontsize=13)
    ax.legend(loc='lower right', fontsize=9)

plt.suptitle('ROC Comparison — Full & Zoomed Views', fontsize=14)
plt.tight_layout()
plt.show()


---
## 22. Final Summary

In [ ]:
print("=" * 75)
print("FINAL MODEL COMPARISON SUMMARY")
print("=" * 75)
print(comparison_df.to_string())

print("\n" + "=" * 75)
print("BEST & WORST MODEL PER CRITERION")
print("=" * 75)
print(f"{'Metric':<15}  {'Best Model':<26}  {'Score':>6}    {'Worst Model':<26}  {'Score':>6}")
print("-" * 75)
for m in ['Accuracy','Sensitivity','Specificity','F1 Score','ROC-AUC']:
    best  = comparison_df[m].idxmax()
    worst = comparison_df[m].idxmin()
    print(f"{m:<15}  {best:<26}  {comparison_df.loc[best,  m]:>6.4f}    "
          f"{worst:<26}  {comparison_df.loc[worst, m]:>6.4f}")

print("\n--- Key Observations ---")
print("• XGBoost achieves the highest overall Accuracy and ROC-AUC,")
print("  indicating the best overall discrimination ability.")
print("• GaussianNB (default) achieves the highest Sensitivity,")
print("  catching more actual No-show cases at the cost of more false alarms.")
print("• GaussianNB (tuned) and XGBoost achieve the highest Specificity,")
print("  rarely misclassifying Show patients as No-show.")
print("• GaussianNB (default) achieves the best F1 Score for the No-show class,")
print("  offering the best precision-recall trade-off for the minority class.")
print("• All models show fair discrimination (AUC 0.67–0.74) due to class imbalance.")


---
# Part 4 — Unsupervised Learning: K-Means Clustering (20 marks)

**Task:** Cluster the California housing dataset using **Median Income**, **Longitude**,
and **Latitude** to create economic segments across different geographic regions (k = 6).

## 23. Load & Explore the Housing Dataset

In [ ]:
housing = pd.read_csv('housing.csv')
print(f"Shape: {housing.shape}")
print("\nColumn descriptions:")
print(housing.dtypes)
print("\nBasic statistics:")
housing.describe().round(3)

In [ ]:
print("Missing values:", housing.isnull().sum().sum())
print("\nSelected features for clustering:")
print(housing[['MedInc', 'Latitude', 'Longitude']].describe().round(3))

## 24. Feature Selection & Standardisation

In [ ]:
from sklearn.preprocessing import StandardScaler

# Select the three attributes specified in the task
features = ['MedInc', 'Latitude', 'Longitude']
X_clust  = housing[features].copy()

# Standardise so that income differences don't dominate geographic distances
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_clust)

print("Original feature ranges:")
for col in features:
    print(f"  {col:12s}: {X_clust[col].min():.3f}  –  {X_clust[col].max():.3f}")

print("\nStandardised feature stats (mean≈0, std≈1):")
print(pd.DataFrame(X_scaled, columns=features).describe().round(3))

## 25. K-Means Clustering  (k = 6)

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=6, random_state=42, n_init=10)
housing['Cluster'] = km.fit_predict(X_scaled)

print(f"K-Means converged in {km.n_iter_} iterations")
print(f"Inertia (within-cluster SSE): {km.inertia_:.2f}")

print("\nCluster sizes:")
sizes = housing['Cluster'].value_counts().sort_index()
print(sizes)

# Centroids back in original scale
centroids_orig = scaler.inverse_transform(km.cluster_centers_)
centroids_df   = pd.DataFrame(centroids_orig,
                               columns=features,
                               index=[f'Cluster {i}' for i in range(6)])
centroids_df.index.name = ''
print("\nCluster centroids (original scale):")
print(centroids_df.round(3))

## 26. Cluster Profile — Summary Statistics

In [ ]:
cluster_stats = (housing.groupby('Cluster')[features + ['MedHouseVal']]
                         .agg(['mean','std'])
                         .round(3))

print("Per-cluster mean and std for each attribute:\n")
print(cluster_stats.to_string())

# Compact mean-only table for quick comparison
mean_table = housing.groupby('Cluster')[features + ['MedHouseVal']].mean().round(3)
mean_table.columns.name = ''
mean_table.index = [f'Cluster {i}' for i in mean_table.index]
print("\nMean values per cluster:")
print(mean_table.to_string())

## 27. Visualisation — Geographic Economic Clusters

In [ ]:
# Colour palette — one per cluster
palette = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4','#42d4f4']
cluster_labels = [f'Cluster {i}' for i in range(6)]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Left: geographic scatter coloured by cluster ──────────────
ax = axes[0]
for cid in range(6):
    mask = housing['Cluster'] == cid
    ax.scatter(housing.loc[mask, 'Longitude'],
               housing.loc[mask, 'Latitude'],
               c=palette[cid], s=2, alpha=0.5,
               label=cluster_labels[cid])

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude',  fontsize=12)
ax.set_title('K-Means Clusters (k=6)\nGeographic Distribution', fontsize=13)
ax.legend(markerscale=5, loc='upper left', fontsize=9)

# ── Right: scatter coloured by cluster, sized by median income ─
ax2 = axes[1]
sc  = ax2.scatter(housing['Longitude'], housing['Latitude'],
                  c=housing['Cluster'], cmap='tab10',
                  s=housing['MedInc'] * 3, alpha=0.4)
plt.colorbar(sc, ax=ax2, label='Cluster ID')
ax2.set_xlabel('Longitude', fontsize=12)
ax2.set_ylabel('Latitude',  fontsize=12)
ax2.set_title('K-Means Clusters\nPoint size ∝ Median Income', fontsize=13)

plt.suptitle('California Housing — Economic Segments by Region', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Income distribution per cluster (box plots)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plots of MedInc per cluster
housing.boxplot(column='MedInc', by='Cluster',
                ax=axes[0], patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[0].set_title('Median Income Distribution per Cluster')
axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('Median Income')
plt.sca(axes[0]); plt.title('Median Income per Cluster')

# Bar chart — mean MedHouseVal per cluster (economic outcome)
mean_val = housing.groupby('Cluster')['MedHouseVal'].mean().sort_index()
bars = axes[1].bar([f'C{i}' for i in mean_val.index],
                    mean_val.values,
                    color=palette, edgecolor='black')
for bar, val in zip(bars, mean_val.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('Mean Median House Value')
axes[1].set_title('Mean House Value per Cluster')

plt.suptitle('Cluster Economic Profiles', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Centroid map — overlay centroids on the geographic scatter
fig, ax = plt.subplots(figsize=(10, 8))

for cid in range(6):
    mask = housing['Cluster'] == cid
    ax.scatter(housing.loc[mask,'Longitude'],
               housing.loc[mask,'Latitude'],
               c=palette[cid], s=3, alpha=0.35,
               label=f'Cluster {cid}  (n={mask.sum():,})')

# Plot centroids
cx = centroids_df['Longitude'].values
cy = centroids_df['Latitude'].values
ax.scatter(cx, cy, c='black', marker='X', s=200, zorder=5, label='Centroids')
for i, (x, y) in enumerate(zip(cx, cy)):
    ax.annotate(f'C{i}\n${centroids_df.iloc[i]["MedInc"]:.1f}k',
                xy=(x, y), xytext=(x + 0.15, y + 0.15), fontsize=8,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude',  fontsize=12)
ax.set_title('K-Means Clustering (k=6) — California Economic Segments\n'
             'Black ✕ = cluster centroid  |  annotation shows mean income',
             fontsize=13)
ax.legend(markerscale=4, loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## 28. Clustering Summary & Interpretation

In [ ]:
print("=" * 65)
print("K-MEANS CLUSTERING SUMMARY  (k = 6)")
print("=" * 65)
print(f"  Inertia (within-cluster SSE) : {km.inertia_:,.2f}")
print(f"  Iterations to convergence    : {km.n_iter_}")
print()

summary = mean_table.copy()
summary['Size'] = housing['Cluster'].value_counts().sort_index().values
summary['Size %'] = (summary['Size'] / len(housing) * 100).round(1)
print(summary[['Size','Size %','MedInc','Latitude','Longitude','MedHouseVal']].to_string())

print()
hi_inc = summary['MedInc'].idxmax()
lo_inc = summary['MedInc'].idxmin()
print(f"  Highest-income cluster : {hi_inc}  (mean MedInc = {summary.loc[hi_inc,'MedInc']:.2f})")
print(f"  Lowest-income cluster  : {lo_inc}  (mean MedInc = {summary.loc[lo_inc,'MedInc']:.2f})")
print()
print("Interpretation:")
print("  • Clusters separate coastal high-income regions (Bay Area, LA)")
print("    from inland lower-income agricultural/suburban zones.")
print("  • Higher MedInc clusters correlate with higher MedHouseVal,")
print("    confirming income as a strong driver of housing prices.")
print("  • Geographic proximity (Lat/Long) groups spatially coherent")
print("    economic zones even without administrative boundaries.")

---
# Part 4b — DBSCAN: Theoretical Analysis & Point Classification (20 marks)

**Parameters:**
- Similarity threshold = **0.8** (two points are neighbours if similarity ≥ 0.8)  
- Equivalent eps = **0.2** on the dissimilarity matrix (dissimilarity = 1 − similarity)  
- **MinPts ≥ 2** — a point is *core* if its ε-neighbourhood contains ≥ 2 points **including itself** (standard DBSCAN definition)

## 29. Similarity & Dissimilarity Matrices

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

points = ['p1', 'p2', 'p3', 'p4', 'p5']

# Similarity matrix (given)
S = np.array([
    [1.00, 0.10, 0.41, 0.55, 0.35],
    [0.10, 1.00, 0.64, 0.47, 0.98],
    [0.41, 0.64, 1.00, 0.44, 0.85],
    [0.55, 0.47, 0.44, 1.00, 0.76],
    [0.35, 0.98, 0.85, 0.76, 1.00],
])

D = 1 - S  # dissimilarity matrix

sim_df  = pd.DataFrame(S, index=points, columns=points)
dist_df = pd.DataFrame(D, index=points, columns=points)

print("=== SIMILARITY MATRIX ===")
print(sim_df.round(2))
print("\n=== DISSIMILARITY MATRIX (D = 1 − S) ===")
print(dist_df.round(2))

In [ ]:
# Heat-map of both matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (data, title, thresh, cbar_lbl) in zip(axes, [
    (sim_df,  'Similarity Matrix\n(neighbours if ≥ 0.8)', 0.8,  'Similarity'),
    (dist_df, 'Dissimilarity Matrix\n(neighbours if ≤ 0.2)', 0.2, 'Dissimilarity'),
]):
    im = ax.imshow(data.values, cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(5)); ax.set_xticklabels(points)
    ax.set_yticks(range(5)); ax.set_yticklabels(points)
    ax.set_title(title, fontsize=12)
    plt.colorbar(im, ax=ax, label=cbar_lbl)
    for i in range(5):
        for j in range(5):
            val = data.values[i, j]
            in_eps = (val >= thresh) if cbar_lbl == 'Similarity' else (val <= thresh)
            txt_clr = 'navy' if in_eps and i != j else 'black'
            weight  = 'bold' if in_eps and i != j else 'normal'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=9, color=txt_clr, fontweight=weight)

plt.suptitle('DBSCAN Input Matrices  (bold = within ε-neighbourhood)', fontsize=13)
plt.tight_layout()
plt.show()

## 30. ε-Neighbourhood Analysis  (step-by-step)

In [ ]:
sim_thresh = 0.8   # similarity threshold
MinPts     = 2     # includes the point itself (standard DBSCAN)

print(f"Parameters: similarity ≥ {sim_thresh}  |  MinPts ≥ {MinPts} (self included)")
print("=" * 60)

neighborhoods = {}
for i, pi in enumerate(points):
    nbrs = [points[j] for j in range(len(points)) if S[i, j] >= sim_thresh]
    neighborhoods[pi] = nbrs

    pairs = []
    for nb in nbrs:
        j = points.index(nb)
        pairs.append(f"{pi}-{nb}: sim={S[i,j]:.2f}")
    print(f"\nN_ε({pi}) = {nbrs}  →  |N_ε({pi})| = {len(nbrs)}")
    print(f"  Qualifying pairs : {', '.join(pairs)}")
    status = f"CORE  (|N| = {len(nbrs)} ≥ {MinPts})" if len(nbrs) >= MinPts else f"not core (|N| = {len(nbrs)} < {MinPts})"
    print(f"  → {status}")

## 31. Point Classification — Core, Border & Noise

In [ ]:
# Classify each point
core_pts   = [p for p in points if len(neighborhoods[p]) >= MinPts]
non_core   = [p for p in points if p not in core_pts]
border_pts = [p for p in non_core
              if any(p in neighborhoods[c] for c in core_pts)]
noise_pts  = [p for p in non_core if p not in border_pts]

print("=" * 60)
print("DBSCAN POINT CLASSIFICATION")
print("=" * 60)
print(f"  Core points   : {core_pts}")
print(f"  Border points : {border_pts if border_pts else 'None'}")
print(f"  Noise points  : {noise_pts}")
print()

# Summary table
rows = []
for p in points:
    if p in core_pts:
        label   = 'Core'
        reason  = f"|N_ε({p})| = {len(neighborhoods[p])} ≥ {MinPts}"
    elif p in border_pts:
        nbr_cores = [c for c in core_pts if p in neighborhoods[c]]
        label   = 'Border'
        reason  = f"not core, but in N_ε of {nbr_cores}"
    else:
        label   = 'Noise'
        reason  = f"|N_ε({p})| = {len(neighborhoods[p])} < {MinPts}, not in any core neighbourhood"
    rows.append({'Point': p, 'N_ε (self included)': neighborhoods[p],
                 'Count': len(neighborhoods[p]), 'Classification': label, 'Reason': reason})

result_df = pd.DataFrame(rows).set_index('Point')
print(result_df.to_string())

print()
print(f"  → Cluster 1: {sorted(set(core_pts + border_pts))}  (core + border points)")
print(f"  → Outliers : {noise_pts}")

## 32. Step-by-Step Explanation

### DBSCAN Definitions (recap)

| Term | Definition |
|---|---|
| **ε-neighbourhood N_ε(p)** | All points (including p itself) with similarity ≥ 0.8 |
| **Core point** | \|N_ε(p)\| ≥ MinPts — dense enough to start/expand a cluster |
| **Border point** | Not core, but lies within N_ε of at least one core point |
| **Noise point** | Neither core nor border — isolated outlier |

---

### Analysis of each point

**p1** — `N_ε(p1) = {p1}` (|N| = 1)  
- sim(p1, p2) = 0.10, sim(p1, p3) = 0.41, sim(p1, p4) = 0.55, sim(p1, p5) = 0.35 — all < 0.8  
- Only neighbour is itself → |N_ε| = 1 < 2 → **not core**  
- p1 does not appear in any core point's neighbourhood → **NOISE** ✗

**p2** — `N_ε(p2) = {p2, p5}` (|N| = 2)  
- sim(p2, p5) = **0.98 ≥ 0.8** → p5 qualifies  
- |N_ε| = 2 ≥ 2 → **CORE** ★

**p3** — `N_ε(p3) = {p3, p5}` (|N| = 2)  
- sim(p3, p5) = **0.85 ≥ 0.8** → p5 qualifies  
- |N_ε| = 2 ≥ 2 → **CORE** ★

**p4** — `N_ε(p4) = {p4}` (|N| = 1)  
- sim(p4, p1)=0.55, sim(p4, p2)=0.47, sim(p4, p3)=0.44, sim(p4, p5)=0.76 — all < 0.8  
- Only neighbour is itself → |N_ε| = 1 < 2 → **not core**  
- p4 does not appear in any core point's neighbourhood → **NOISE** ✗

**p5** — `N_ε(p5) = {p2, p3, p5}` (|N| = 3)  
- sim(p5, p2) = **0.98 ≥ 0.8**, sim(p5, p3) = **0.85 ≥ 0.8** → both qualify  
- |N_ε| = 3 ≥ 2 → **CORE** ★

---

### Border check for non-core points p1, p4

Core neighbourhoods: N_ε(p2)={p2,p5}, N_ε(p3)={p3,p5}, N_ε(p5)={p2,p3,p5}

- **p1** not in any core neighbourhood → **Noise**  
- **p4** not in any core neighbourhood → **Noise**  
- *(No border points exist in this dataset)*

---

### Result

| Classification | Points |
|---|---|
| **Core** | p2, p3, p5 |
| **Border** | *(none)* |
| **Noise** | p1, p4 |
| **Cluster 1** | {p2, p3, p5} — all mutually density-connected through p5 |

## 33. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: neighbourhood graph ─────────────────────────────────
ax = axes[0]
ax.set_xlim(-0.5, 4.5); ax.set_ylim(-0.5, 3.5)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('ε-Neighbourhood Graph\n(edges = similarity ≥ 0.8)', fontsize=12)

# Node positions
pos = {'p1': (0, 0), 'p2': (1, 3), 'p3': (3, 3), 'p4': (4, 0), 'p5': (2, 1.5)}
node_color = {'p1':'#d62728','p2':'#2ca02c','p3':'#2ca02c','p4':'#d62728','p5':'#2ca02c'}
node_label = {'p1':'NOISE','p2':'CORE','p3':'CORE','p4':'NOISE','p5':'CORE'}

# Draw edges (sim >= 0.8, excluding self-loops)
for i, pi in enumerate(points):
    for j, pj in enumerate(points):
        if j > i and S[i, j] >= 0.8:
            x1, y1 = pos[pi]; x2, y2 = pos[pj]
            ax.plot([x1, x2], [y1, y2], 'k-', lw=2, zorder=1)
            mx, my = (x1+x2)/2, (y1+y2)/2
            ax.text(mx, my+0.12, f'{S[i,j]:.2f}', ha='center',
                    va='bottom', fontsize=8, color='dimgray')

# Draw nodes
for p, (x, y) in pos.items():
    circ = plt.Circle((x, y), 0.35, color=node_color[p],
                        ec='black', lw=1.5, zorder=2)
    ax.add_patch(circ)
    ax.text(x, y+0.02, p, ha='center', va='center',
            fontsize=11, fontweight='bold', color='white', zorder=3)
    ax.text(x, y-0.55, node_label[p], ha='center', va='top',
            fontsize=8, style='italic', color=node_color[p])

legend_els = [mpatches.Patch(color='#2ca02c', label='Core point'),
              mpatches.Patch(color='#d62728', label='Noise point')]
ax.legend(handles=legend_els, loc='lower center', fontsize=9)

# ── Right: classification summary table as heatmap ────────────
ax2 = axes[1]
ax2.axis('off')
ax2.set_title('Classification Summary', fontsize=12)

cell_data = []
col_labels = ['|N_ε| (self incl.)', 'Qualifies (≥2)?', 'In core N_ε?', 'Classification']
for p in points:
    n   = len(neighborhoods[p])
    q   = '✓' if n >= MinPts else '✗'
    inc = '✓' if any(p in neighborhoods[c] for c in core_pts) else '✗'
    cls = ('CORE' if p in core_pts
           else 'BORDER' if p in border_pts
           else 'NOISE')
    cell_data.append([f'{neighborhoods[p]}  (n={n})', q, inc, cls])

colors_table = []
for row in cell_data:
    cls = row[-1]
    bg  = '#c3e6cb' if cls == 'CORE' else '#f5c6cb' if cls == 'NOISE' else '#ffeeba'
    colors_table.append(['#f8f9fa', '#f8f9fa', '#f8f9fa', bg])

tbl = ax2.table(
    cellText   = cell_data,
    rowLabels  = points,
    colLabels  = col_labels,
    cellColours= colors_table,
    loc        = 'center',
    cellLoc    = 'center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.4, 2.0)

plt.suptitle('DBSCAN Analysis  (sim ≥ 0.8, MinPts = 2)', fontsize=14)
plt.tight_layout()
plt.show()